Import JSON Device Recall Data

In [36]:
import pandas as pd
import json
from sklearn.feature_extraction.text import CountVectorizer
from collections import Counter
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



DataFrame shape: (57511, 35)
Columns: Index(['cfres_id', 'product_res_number', 'event_date_initiated',
       'event_date_posted', 'recall_status', 'res_event_number',
       'product_code', 'product_description', 'code_info', 'recalling_firm',
       'address_1', 'city', 'state', 'postal_code', 'additional_info_contact',
       'reason_for_recall', 'root_cause_description', 'action',
       'product_quantity', 'distribution_pattern', 'openfda.k_number',
       'openfda.registration_number', 'openfda.fei_number',
       'openfda.device_name', 'openfda.medical_specialty_description',
       'openfda.regulation_number', 'openfda.device_class', 'address_2',
       'event_date_terminated', 'firm_fei_number',
       'other_submission_description', 'k_numbers', 'pma_numbers',
       'event_date_created', 'openfda.pma_number'],
      dtype='object')


In [ ]:
json_path = "/projectnb/sparkgrp/dd4g-muckrock-patient-safety/device-recall-0001-of-0001.json"

# Load JSON
with open(json_path, 'r') as f:
    data = json.load(f)

#print(type(data)) 
#print(list(data.keys())

df = pd.json_normalize(data['results'])
#print("DataFrame shape:", df.shape)
#print("Columns:", df.columns)


In [37]:
df.head()

,cfres_id,product_res_number,event_date_initiated,event_date_posted,recall_status,res_event_number,product_code,product_description,code_info,recalling_firm,...,openfda.regulation_number,openfda.device_class,address_2,event_date_terminated,firm_fei_number,other_submission_description,k_numbers,pma_numbers,event_date_created,openfda.pma_number
0,218307,Z-1472-2026,2026-01-21,2026-02-26,"Open, Classified",98332,FNM,"Adapt Pump, Model/Catalog Number: 61600200-Ada...",UDI-DI 0084569904914; All serial numbers with ...,Agiliti Health Ellis,...,880.5550,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,218406,Z-1462-2026,2026-01-14,2026-02-24,"Open, Classified",98303,KPL,"PIE PAK\nModels: P2HC-A, P2HC-S, P2HC",All Lots\nUDI:,Edermy LLC,...,876.5220,2,Ste A,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,105069,Z-0389-2012,2011-09-19,2012-01-11,Terminated,60344,LFQO,Magnus Hybrid operating table columns 1180.01A...,"510 k EXEMPT 1180.01A1- Lot #0002, 0004, 0005...",Maquet Inc.,...,NaN,NaN,NaN,2011-12-08,NaN,NaN,NaN,NaN,NaN,NaN
3,105700,Z-0485-2012,2011-11-28,2012-01-11,Terminated,60528,RER,Raymond Laser alignment guide - Model #1096597...,Model #1096597/002,The Raymond Corporation S Canal,...,,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,131624,Z-0604-2015,2014-11-17,2014-12-12,Terminated,69838,KOG,"PKG, OBTURATOR, 5.5MM FOR HASSON CANNULA, P/N ...",37302 37858 39543 45622 48347 48903 49147 4914...,Stryker Endoscopy,...,NaN,NaN,NaN,2015-07-17,2936485,NaN,NaN,NaN,NaN,NaN


Identify Unique Product Codes and IDs

In [39]:
print(df['product_code'].unique()[:10])
print(df['cfres_id'].unique()[:10])

['FNM' 'KPL' 'LFQO' 'RER' 'KOG' 'N/A' 'RCE' 'REM' 'RHP' 'MIE']
['218307' '218406' '105069' '105700' '131624' '131797' '131743' '131693'
 '131625' '131848']


Basic EDA (explore how many rows contain issues involving batteries, for instance)

In [40]:
battery_rows = df[df['reason_for_recall'].str.contains("battery", case=False, na=False)]

print(battery_rows.shape)
print(battery_rows.head()) 

(920, 35)
    cfres_id product_res_number event_date_initiated event_date_posted  \
124   181608        Z-0518-2021           2020-04-30               NaN   
143   172735        Z-1773-2019           2019-05-10               NaN   
495   103146        Z-0142-2012           2011-09-23        2011-10-31   
621   112481        Z-0049-2013           2012-09-25        2012-10-11   
862    64398        Z-0051-2008           2007-03-16        2007-11-24   

        recall_status res_event_number product_code  \
124  Open, Classified            85711          N/A   
143        Terminated            82803          LPA   
495        Terminated            59582          DRY   
621        Terminated            63079          KFM   
862        Terminated            44793          LWP   

                                   product_description  \
124  MiniMed 640G, insulin pump, Ref/Model # MMT-17...   
143  CardioCommand, Inc. Transesophageal Cardiac Pa...   
495  CDI Blood Parameter Monitoring Syst

In [41]:
#account for nulls 
df['reason_for_recall'] = df['reason_for_recall'].fillna('')

#how many rows contain batteries
battery_count = df['reason_for_recall'].str.contains('battery', case=False).sum()
print(f"Rows containing 'battery': {battery_count}")

Rows containing 'battery': 920


In [42]:
vectorizer = CountVectorizer(ngram_range=(2,2), stop_words='english')
X = vectorizer.fit_transform(df['reason_for_recall'])

#sum columns directly on sparse matrix
sums = X.sum(axis=0)  
sums_array = sums.A1 

bigrams = Counter(dict(zip(vectorizer.get_feature_names_out(), sums_array)))
print(bigrams.most_common(20))

[('instructions use', np.int64(1165)), ('sterile barrier', np.int64(1099)), ('recall initiated', np.int64(1010)), ('expiration date', np.int64(961)), ('kits contain', np.int64(768)), ('compromising sterility', np.int64(756)), ('package integrity', np.int64(719)), ('shelf life', np.int64(690)), ('non sterile', np.int64(666)), ('use ifu', np.int64(634)), ('specific lots', np.int64(617)), ('received complaints', np.int64(614)), ('convenience kits', np.int64(610)), ('ge healthcare', np.int64(586)), ('firm received', np.int64(573)), ('lot number', np.int64(525)), ('received reports', np.int64(513)), ('ethylene oxide', np.int64(492)), ('voluntary recall', np.int64(470)), ('product sterility', np.int64(466))]


Import FDA Problem Codes Dataset

In [43]:
foi_file = "/projectnb/sparkgrp/dd4g-muckrock-patient-safety/foiclass.txt" 

In [44]:
product_codes = pd.read_csv(foi_file, sep="|", dtype=str, encoding="cp1252")

In [45]:
print(product_codes.head())
print(product_codes.columns)

  REVIEW_PANEL MEDICALSPECIALTY PRODUCTCODE  \
0           AN               AN         BRW   
1           AN               AN         BRX   
2           AN               AN         BRY   
3           AN               AN         BSE   
4           AN               AN         BSF   

                             DEVICENAME DEVICECLASS UNCLASSIFIED_REASON  \
0                     Protector, Dental           1                 NaN   
1                     Stool, Anesthesia           1                 NaN   
2   Cabinet, Table And Tray, Anesthesia           1                 NaN   
3  Analyzer, Gas, Helium, Gaseous-Phase           2                 NaN   
4              Absorber, Carbon-Dioxide           1                 NaN   

  GMPEXEMPTFLAG THIRDPARTYFLAG REVIEWCODE REGULATIONNUMBER SUBMISSION_TYPE_ID  \
0             N              N        NaN         868.5820                  4   
1             N              N        NaN         868.6700                  4   
2             N        

In [46]:
print(product_codes.columns)

Index(['REVIEW_PANEL', 'MEDICALSPECIALTY', 'PRODUCTCODE', 'DEVICENAME',
       'DEVICECLASS', 'UNCLASSIFIED_REASON', 'GMPEXEMPTFLAG', 'THIRDPARTYFLAG',
       'REVIEWCODE', 'REGULATIONNUMBER', 'SUBMISSION_TYPE_ID', 'DEFINITION',
       'PHYSICALSTATE', 'TECHNICALMETHOD', 'TARGETAREA', 'Implant_Flag',
       'Life_Sustain_support_flag', 'SummaryMalfunctionReporting'],
      dtype='object')


In [47]:
#product code is string
df['product_code'] = df['product_code'].astype(str)

#fda table is string
product_codes['PRODUCTCODE'] = product_codes['PRODUCTCODE'].astype(str)

#merged recall dataset
merged_df = df.merge(
    product_codes, 
    left_on='product_code', 
    right_on='PRODUCTCODE', 
    how='left'
)

#merged columns
print(merged_df[['product_code', 'product_description', 'DEVICENAME', 'DEVICECLASS']].head())

  product_code                                product_description  \
0          FNM  Adapt Pump, Model/Catalog Number: 61600200-Ada...   
1          KPL              PIE PAK\nModels: P2HC-A, P2HC-S, P2HC   
2         LFQO  Magnus Hybrid operating table columns 1180.01A...   
3          RER  Raymond Laser alignment guide - Model #1096597...   
4          KOG  PKG, OBTURATOR, 5.5MM FOR HASSON CANNULA, P/N ...   

                                      DEVICENAME DEVICECLASS  
0  Mattress, Air Flotation, Alternating Pressure           2  
1                      Colonic Irrigation System           2  
2                                            NaN         NaN  
3                        Alignment Laser Product           N  
4                                            NaN         NaN  


In [48]:
print(merged_df.columns)
print(df.columns)


Index(['cfres_id', 'product_res_number', 'event_date_initiated',
       'event_date_posted', 'recall_status', 'res_event_number',
       'product_code', 'product_description', 'code_info', 'recalling_firm',
       'address_1', 'city', 'state', 'postal_code', 'additional_info_contact',
       'reason_for_recall', 'root_cause_description', 'action',
       'product_quantity', 'distribution_pattern', 'openfda.k_number',
       'openfda.registration_number', 'openfda.fei_number',
       'openfda.device_name', 'openfda.medical_specialty_description',
       'openfda.regulation_number', 'openfda.device_class', 'address_2',
       'event_date_terminated', 'firm_fei_number',
       'other_submission_description', 'k_numbers', 'pma_numbers',
       'event_date_created', 'openfda.pma_number', 'REVIEW_PANEL',
       'MEDICALSPECIALTY', 'PRODUCTCODE', 'DEVICENAME', 'DEVICECLASS',
       'UNCLASSIFIED_REASON', 'GMPEXEMPTFLAG', 'THIRDPARTYFLAG', 'REVIEWCODE',
       'REGULATIONNUMBER', 'SUBMISSION_T

In [49]:
#turn device class field into strings 
merged_df['DEVICECLASS'] = merged_df['DEVICECLASS'].astype(str)

In [50]:
#check which product codes were not matched during merge 
unmatched = merged_df[merged_df['DEVICENAME'].isna()]['product_code'].unique()
print("Unmatched product codes:", unmatched)

Unmatched product codes: ['LFQO' 'KOG' 'N/A' '90LMA' '74DWF' 'nan' 'FRN--' 'rad' 'U28' 'CCH' 'BTE'
 'MPH' 'K--DD' 'dxg' 'NWR' 'LDT' 'ND' 'LNX' 'OWG' 'L22' 'NED' 'GBK' 'NA'
 '---' 'F' 'ndr' 'NPD' 'D' 'muj' 'cac' 'cck' 'NSX' 'MJQ' 'LNN' 'PRD' 'GBR'
 ':' 'KPF--' 'CAJ' 'LMO' 'DJR--' '74' 'FFK--' 'LNO' 'M' 'OGX' 'N/a' 'OYT'
 'MEW' 'NES' 'LXL' 'O-FY' 'H--NN' 'MFC' 'gkz' 'LDH' 'GAB--' 'JAJ' 'IZO--'
 'CGN--' 'LHI--' 'MKI' 'CEC--' 'DKJ--' 'JD-01' 'NFN' 'MDV' 'DS-22' 'EDI'
 'L--22' 'FSG' 'max' '80FMZ' 'PIU' 'JZL' 'J' 'NEP--' 'frn' 'KPS--' 'LID'
 'I--TI' 'CPK' 'SFB' 'ZZZ' 'NWD' 'EXEMP' 'FKX--' 'BBT' 'NCP' 'L' '79F'
 'LTK--' 'LPD' 'MPN--' 'CXT' 'EDH' 'GER' 'U--19' 'LLT' '-KPS' 'KPG'
 'GAA41' 'IZL--' 'NPN' 'LYW' 'KKX--' 'JQW--' 'HAP' 'ITF' '-JTC' '91' 'gbx'
 'LLC' 'OPU' 'LOS' 'ORO' 'GEL' 'LYB' '--' 'G-IO' 'GCB--' 'jjy' 'GKI'
 'KPR--' '-IYE' 'OMO' 'rce' 'DZi' 'LOR' 'LOM--' '6885' '74GBK' 'KMM' 'OOC'
 'BGK' '000' 'MXE' 'MFT' 'hpz' 'XLH' 'cbk' 'PNH' 'mkj' 'DIC--' 'MLM--'
 'LS-99' 'DS-21' 'NXS' 'lZE' '

In [51]:
#relevant cols 
keep_cols = ['cfres_id', 'product_code', 'product_description', 
             'reason_for_recall', 'DEVICENAME', 'DEVICECLASS']

#cleaned df
clean_df = merged_df[keep_cols].copy()

print(clean_df.head())
print(clean_df.columns)

  cfres_id product_code                                product_description  \
0   218307          FNM  Adapt Pump, Model/Catalog Number: 61600200-Ada...   
1   218406          KPL              PIE PAK\nModels: P2HC-A, P2HC-S, P2HC   
2   105069         LFQO  Magnus Hybrid operating table columns 1180.01A...   
3   105700          RER  Raymond Laser alignment guide - Model #1096597...   
4   131624          KOG  PKG, OBTURATOR, 5.5MM FOR HASSON CANNULA, P/N ...   

                                   reason_for_recall  \
0  Agiliti identified an issue where the Adapt Pu...   
1                             Lack of 510K clearance   
2  A component of the Magnus hybrid OR system fai...   
3  Raymond shipped a laser alignment guide as an ...   
4  The parameters provided in the Laparoscopic Ma...   

                                      DEVICENAME DEVICECLASS  
0  Mattress, Air Flotation, Alternating Pressure           2  
1                      Colonic Irrigation System           2  
2    

In [52]:
clean_df.rename(columns={
    'DEVICENAME': 'device_name',
    'DEVICECLASS': 'device_class'
}, inplace=True)

print(clean_df.columns)

Index(['cfres_id', 'product_code', 'product_description', 'reason_for_recall',
       'device_name', 'device_class'],
      dtype='object')


In [54]:
clean_df['reason_for_recall'] = clean_df['reason_for_recall'].fillna("").str.lower()

In [55]:
problem_codes_file = "/projectnb/sparkgrp/dd4g-muckrock-patient-safety/extracted_problems/deviceproblemcodes.csv"
df_pc = pd.read_csv(problem_codes_file)

In [56]:
print(df_pc.head())
print(df_pc.columns)

   FDA_CODE                      TERM NCIT_CODE IMDRF_CODE
0         1     Old To Be Deactivated       NaN        NaN
1      1001   Failure To Run On AC/DC       NaN        NaN
2      1002                  Abnormal       NaN        NaN
3      1003                Absorption       NaN        NaN
4      1005  Measurements, inaccurate       NaN        NaN
Index(['FDA_CODE', 'TERM', 'NCIT_CODE', 'IMDRF_CODE'], dtype='object')


In [57]:
df_pc['FDA_CODE'] = df_pc['FDA_CODE'].astype(str)

In [58]:
df_pc['TERM'] = df_pc['TERM'].fillna("").str.lower()

In [59]:
recall_texts = clean_df['reason_for_recall'].tolist()
problem_terms = df_pc['TERM'].tolist()

all_text = recall_texts + problem_terms

In [60]:
vectorizer = TfidfVectorizer(stop_words='english')

tfidf_matrix = vectorizer.fit_transform(all_text)

In [61]:
recall_vectors = tfidf_matrix[:len(recall_texts)]
term_vectors = tfidf_matrix[len(recall_texts):]

In [62]:
similarity_matrix = cosine_similarity(recall_vectors, term_vectors)

In [63]:
#setting threshold, 0.40??? less preczse, more broad 

best_match_index = similarity_matrix.argmax(axis=1)
best_scores = similarity_matrix.max(axis=1)

threshold = 0.25

In [ ]:
best_scores[:20]

array([0.13571234, 0.27584258, 0.35678356, 0.10270102, 0.34994653,
       0.34994653, 0.34994653, 0.34994653, 0.34994653, 0.34994653,
       0.34994653, 0.34994653, 0.34994653, 0.34994653, 0.34994653,
       0.34994653, 0.34994653, 0.34994653, 0.34994653, 0.34994653])

In [64]:
best_match_index = similarity_matrix.argmax(axis=1)

clean_df['similarity_score'] = best_scores

clean_df['FDA_CODE'] = None
clean_df['TERM'] = None

mask = best_scores >= threshold

clean_df.loc[mask, 'FDA_CODE'] = df_pc.iloc[best_match_index[mask]]['FDA_CODE'].values
clean_df.loc[mask, 'TERM'] = df_pc.iloc[best_match_index[mask]]['TERM'].values

In [65]:
clean_df[['reason_for_recall','FDA_CODE','TERM']]

,reason_for_recall,FDA_CODE,TERM
0,agiliti identified an issue where the adapt pu...,None,None
1,lack of 510k clearance,4065,lack of effect
2,a component of the magnus hybrid or system fai...,1299,hybrid failure
3,raymond shipped a laser alignment guide as an ...,None,None
4,the parameters provided in the laparoscopic ma...,1142,failure to cycle
...,...,...,...
57506,philips has identified that in a limited numbe...,None,None
57507,access port system tray may develop cracks po...,None,None
57508,medline kits contain b. braun iv administratio...,None,None
57509,medline kits containing stryker's strykeflow 2...,None,None


In [66]:
#merges problem code dataset with recall txt - no cosine similarity, just appending extra info 
final_df = clean_df.merge(df_pc, on='FDA_CODE', how='left')

In [67]:
final_df = final_df.drop(columns=['TERM_x'])
final_df = final_df.rename(columns={'TERM_y':'problem_term'})

In [68]:
final_df['problem_term'].value_counts().head(20)

problem_term
sterility                       1013
device packaging compromised     570
expiration date error            519
performance                      409
port leak(s)                     308
screw head(s), incorrect         295
seal, incorrect                  293
shelf life exceeded              270
crack                            239
tube(s), defective               223
off-label use                    222
blood in tubing                  222
failure to cycle                 216
false negative result            215
reservoir                        214
air leak                         201
inadequate storage               200
moisture damage                  199
break                            191
no pressure                      181
Name: count, dtype: int64

In [73]:
final_df['problem_term'].unique()

array([nan, 'lack of effect', 'hybrid failure', 'failure to cycle',
       'radiation leak', 'human factors issue',
       'incorrect or inadequate test results',
       'locking mechanism failure', 'motor failure',
       'application network problem', 'malfunction',
       'grounding malfunction',
       'false reading from device non-compliance', 'failure to calibrate',
       'moisture damage', 'lead(s), burn(s) from', 'lead(s), shock from',
       'device packaging compromised', 'shielding failure',
       'valve(s), failure of', 'difficult to insert',
       'false positive result', 'failure to shut off',
       'filtration problem', 'safety interlock(s) inadequate',
       'probe failure', 'insulation failure', 'short fill',
       'unit inactivated', 'expiration date error',
       'solder joint fracture', 'laser output, unintended', 'mislabeled',
       'blood in tubing', 'sterility', 'battery failure', 'off-label use',
       'failure to conduct', 'normal', 'charging problem'

In [70]:
clean_df['similarity_score'].describe()

count    57511.000000
mean         0.239033
std          0.121919
min          0.000000
25%          0.157825
50%          0.224107
75%          0.298726
max          1.000000
Name: similarity_score, dtype: float64

In [75]:
total = len(final_df)
matched = final_df['problem_term'].notna().sum()

print("Total recalls:", total)
print("Matched recalls:", matched)
print("Coverage:", matched / total)

Total recalls: 57511
Matched recalls: 23049
Coverage: 0.40077550381666116


In [88]:
final_df['problem_term'].value_counts().head(15)

problem_term
sterility                       1013
device packaging compromised     570
expiration date error            519
performance                      409
port leak(s)                     308
screw head(s), incorrect         295
seal, incorrect                  293
shelf life exceeded              270
crack                            239
tube(s), defective               223
off-label use                    222
blood in tubing                  222
failure to cycle                 216
false negative result            215
reservoir                        214
Name: count, dtype: int64

In [82]:
final_df

,cfres_id,product_code,product_description,reason_for_recall,device_name,device_class,similarity_score,FDA_CODE,problem_term,NCIT_CODE,IMDRF_CODE
0,218307,FNM,"Adapt Pump, Model/Catalog Number: 61600200-Ada...",agiliti identified an issue where the adapt pu...,"Mattress, Air Flotation, Alternating Pressure",2,0.135712,None,NaN,NaN,NaN
1,218406,KPL,"PIE PAK\nModels: P2HC-A, P2HC-S, P2HC",lack of 510k clearance,Colonic Irrigation System,2,0.275843,4065,lack of effect,C187247,A0105
2,105069,LFQO,Magnus Hybrid operating table columns 1180.01A...,a component of the magnus hybrid or system fai...,NaN,nan,0.356784,1299,hybrid failure,NaN,NaN
3,105700,RER,Raymond Laser alignment guide - Model #1096597...,raymond shipped a laser alignment guide as an ...,Alignment Laser Product,N,0.102701,None,NaN,NaN,NaN
4,131624,KOG,"PKG, OBTURATOR, 5.5MM FOR HASSON CANNULA, P/N ...",the parameters provided in the laparoscopic ma...,NaN,nan,0.349947,1142,failure to cycle,C63187,A050703
...,...,...,...,...,...,...,...,...,...,...,...
57506,217626,OWB,Product Name: Allura Xper FD10C;\nModel number...,philips has identified that in a limited numbe...,Interventional Fluoroscopic X-Ray System,2,0.203881,None,NaN,NaN,NaN
57507,217850,NAY,"da Vinci SP, SP Access Port Kit, Large Incisi...",access port system tray may develop cracks po...,"System, Surgical, Computer Controlled Instrument",2,0.245580,None,NaN,NaN,NaN
57508,217742,OJG,Medline Kits containing B. Braun IV Administra...,medline kits contain b. braun iv administratio...,Neurological Tray,1,0.148730,None,NaN,NaN,NaN
57509,217876,FDE,Medline medical procedure convenience kits lab...,medline kits containing stryker's strykeflow 2...,Laparoscopy Kit,2,0.243951,None,NaN,NaN,NaN


In [87]:
final_df[['cfres_id','reason_for_recall','FDA_CODE','problem_term']].sample(20)

,cfres_id,reason_for_recall,FDA_CODE,problem_term
23475,110836,distal part of the anchor may break on inserti...,1069,break
13104,111751,there is potential for the sterility of the pr...,1595,sterility
24331,67897,slow deflation or no deflation,1149,deflation problem
35955,95190,there is a potential for delamination in one l...,2904,delamination
25954,140891,user settings were no longer on the monitor's ...,1407,monitor failure
32843,162149,this correction is being initiated based on a ...,None,NaN
44424,204519,masks non-invasive interface for channeling ai...,None,NaN
15274,174503,the protective material used in the manufactur...,None,NaN
53583,51504,unintended gantry movements : ge healthcare ha...,None,NaN
49211,160233,the locking punch guide was not sufficiently g...,None,NaN
